# 02 Data Cleaning

## 1. Objective

The objective of this notebook is to clean and prepare the raw debt-collection datasets for exploratory analysis and predictive modelling. Cleaning decisions will be based on data quality, business context, and the intended prediction task. Significant transformations and exclusions will be documented to maintain reproducibility.

In [11]:
import pandas as pd
import numpy as np

## 2. Load Data

The raw debt-collection datasets are loaded into pandas DataFrames for further inspection and cleaning. The original files are kept unchanged so that all cleaning and transformation steps can be reproduced.

In [12]:
from pathlib import Path

print(Path.cwd())

/Users/jamie-leejoseph/Documents/debt-collection-prediction/notebooks


In [13]:
from pathlib import Path

print(Path("../data").exists())
print(list(Path("../data").iterdir()))

True
[PosixPath('../data/processed'), PosixPath('../data/raw')]


In [14]:
print(list(Path("../data/raw").iterdir()))

[PosixPath('../data/raw/PTP.xlsx'), PosixPath('../data/raw/Matters.xlsx'), PosixPath('../data/raw/Payments.xlsx'), PosixPath('../data/raw/UKZN_EnrichedData2.xlsx'), PosixPath('../data/raw/UKZN_EnrichedData1.xlsx'), PosixPath('../data/raw/CallHistory.xlsx'), PosixPath('../data/raw/EnrichedData_Combined.xlsx')]


In [15]:
matters = pd.read_excel(
    "../data/raw/Matters.xlsx",
    sheet_name="Data"
)

In [16]:
payments = pd.read_excel(
    "../data/raw/Payments.xlsx",
    sheet_name="Data"
)

In [17]:
ptp = pd.read_excel(
    "../data/raw/PTP.xlsx",
    sheet_name="Data"
)

In [18]:
call_history = pd.read_excel(
    "../data/raw/CallHistory.xlsx",
    sheet_name="Data"
)

In [19]:
enriched_combined = pd.read_excel(
    "../data/raw/EnrichedData_Combined.xlsx"
)

In [20]:
print(f"Matters:")
print(f"Rows: {matters.shape[0]:,}")
print(f"Columns: {matters.shape[1]:,}")

Matters:
Rows: 56,179
Columns: 12


In [21]:
print(f"Call History:")
print(f"R0ws: {call_history.shape[0]:,}")
print(f"Columns: {call_history.shape[1]:,}")

Call History:
R0ws: 430,197
Columns: 12


In [22]:
print(f"Payments:")
print(f"Rows: {payments.shape[0]:,}")
print(f"Columns: {payments.shape[1]:,}")

Payments:
Rows: 6,016
Columns: 8


In [23]:
print(f"PTPs:")
print(f"Rows: {ptp.shape[0]:,}")
print(f"Columns: {ptp.shape[1]:,}")


PTPs:
Rows: 9,402
Columns: 16


In [24]:
print(f"Enriched:")
print(f"Rows: {enriched_combined.shape[0]:,}")
print(f"Columnns: {enriched_combined.shape[1]:,}")

Enriched:
Rows: 55,086
Columnns: 517


## 3. Cleaning Preparation

The initial data understanding conducted in the previous notebook identified the structure, data types, missing-value patterns, duplicates, and potential data-quality issues within the datasets. This notebook builds on those findings and focuses on applying and documenting the required cleaning and preparation steps.

In [25]:
matters.info()

<class 'pandas.DataFrame'>
RangeIndex: 56179 entries, 0 to 56178
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   M_IDX             56179 non-null  int64         
 1   HandoverDate      56179 non-null  datetime64[us]
 2   CurrentStatusID   56179 non-null  int64         
 3   CurrentStatus     56179 non-null  str           
 4   PreviousStatusID  1152 non-null   float64       
 5   PreviousStatus    1152 non-null   str           
 6   FirstPaymentDate  3771 non-null   datetime64[us]
 7   ActivationPeriod  3771 non-null   float64       
 8   OpeningBalance    56179 non-null  float64       
 9   CurrentBalance    55240 non-null  float64       
 10  Industry          56179 non-null  str           
 11  DateCreated       56179 non-null  datetime64[us]
dtypes: datetime64[us](3), float64(4), int64(2), str(3)
memory usage: 5.1 MB


In [26]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 6016 entries, 0 to 6015
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   M_IDX                6016 non-null   int64         
 1   PaymentDate          6016 non-null   datetime64[us]
 2   AmountPaid           6016 non-null   float64       
 3   IsPayedAtClient      6016 non-null   int64         
 4   IsDebitOrderPayment  6016 non-null   int64         
 5   PaymentMethodID      6016 non-null   int64         
 6   PaymentDescription   6016 non-null   str           
 7   AgentID              6016 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(5), str(1)
memory usage: 376.1 KB


In [27]:
call_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 430197 entries, 0 to 430196
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   M_IDX               430197 non-null  int64         
 1   AgentID             430197 non-null  int64         
 2   HistoryID           430197 non-null  int64         
 3   CallDate            430197 non-null  datetime64[us]
 4   MinutesOfCall       204066 non-null  float64       
 5   HasRPC              82508 non-null   float64       
 6   CallTypeID          350124 non-null  float64       
 7   CallType            430197 non-null  str           
 8   PTPCreateIndicator  8209 non-null    float64       
 9   TimeCallStart       209016 non-null  str           
 10  TimeCallConfirmRPC  17416 non-null   str           
 11  TimeCallEnded       204066 non-null  str           
dtypes: datetime64[us](1), float64(4), int64(3), str(4)
memory usage: 39.4 MB


In [28]:
ptp.info()

<class 'pandas.DataFrame'>
RangeIndex: 9402 entries, 0 to 9401
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   M_IDX                    9402 non-null   int64         
 1   PTPCreateDate            9402 non-null   datetime64[us]
 2   PaymentMethodID          9402 non-null   int64         
 3   PaymentDescription       9402 non-null   str           
 4   PaymentFrequencyID       9402 non-null   int64         
 5   PaymentFrequency         7567 non-null   str           
 6   FirstPaymentAmount       9241 non-null   float64       
 7   FirstPaymentDate         3228 non-null   datetime64[us]
 8   MonthlyPaymentAmount     9402 non-null   float64       
 9   MonthlyPaymentStartDate  7568 non-null   datetime64[us]
 10  DebitAmount              11 non-null     float64       
 11  CreditAmount             9402 non-null   float64       
 12  ProjectionPaymentDate    9402 non-null   date

In [29]:
enriched_combined.info()

<class 'pandas.DataFrame'>
RangeIndex: 55086 entries, 0 to 55085
Columns: 517 entries, M_IDX to Cell1Score
dtypes: datetime64[us](1), float64(356), int64(10), object(41), str(109)
memory usage: 217.3+ MB


## 4. Matters Cleaning

### 4.1 Duplicate Matters
Duplicate records can affect the reliability of the modelling dataset. The M_IDX field is used to identify whether the same matter appears more than once.

In [30]:
matters["M_IDX"].nunique()


56179

In [31]:
matters["M_IDX"].is_unique

True

**Finding:** Each matter has a unique `M_IDX`. There are 56,179 unique matter IDs across 56,179 rows, so no duplicate matters were identified and no rows were removed at this stage.

In [32]:
matters[matters["CurrentBalance"].isna()]["CurrentStatus"].value_counts()

CurrentStatus
Closed    939
Name: count, dtype: int64

In [33]:
matters[matters["CurrentBalance"].isna()]["OpeningBalance"].describe()

count       939.000000
mean      14687.616539
std       17730.374895
min         144.680000
25%        3715.200000
50%        9142.220000
75%       18844.820000
max      181597.150000
Name: OpeningBalance, dtype: float64

In [34]:
matters.groupby("CurrentStatus")[["OpeningBalance", "CurrentBalance"]].agg(["count", "mean"])

OpeningBalance               CurrentBalance              
                             count          mean          count          mean
CurrentStatus                                                                
Attempting PTP               26474   8370.901968          26474   8486.157391
Authentication                 271  10786.158081            271  10860.610996
Broken PTP                    2817   7329.035279           2817   7384.717636
Closed                       15882  11734.165002          14943  11469.760656
Dispute                          1   3435.280000              1   3616.330000
Follow-up PTP                   70   5319.022143             70   3741.016143
In Progress                    232   9604.242198            232   9601.309741
New Instruction               6341  11512.631798           6341  11536.247649
On Hold                        160   3122.188000            160   2654.922125
Payment Arrangement           3633   9589.225092           3633   9491.445866
Re-opened                      292   1862.664041            292   1808.531952
Request for Closure              6   4274.270000              6   4213.498333

### 4.3 Opening Balance and Current Balance
The relationship between OpeningBalance and CurrentBalance is investigated to identify matters where no change in balance has occurred. These records will be assessed in the context of the business rules and modelling population before any exclusions are made.

In [35]:
equal_balance = matters["OpeningBalance"] == matters["CurrentBalance"]

equal_balance.sum()

np.int64(15604)

In [36]:
matters.loc[equal_balance, "CurrentStatus"].value_counts()

CurrentStatus
Closed                 9515
Attempting PTP         3623
New Instruction         949
Broken PTP              695
Payment Arrangement     428
Re-opened               267
On Hold                  59
In Progress              54
Authentication           12
Follow-up PTP             2
Name: count, dtype: int64

In [37]:
closed_equal_balance = (
    (matters["CurrentStatus"] == "Closed") &
    equal_balance
)

closed_equal_balance.sum()

np.int64(9515)

In [38]:
matters.loc[
    closed_equal_balance,
    "FirstPaymentDate"
].isna().sum()

np.int64(9381)

In [39]:
matters.loc[
    closed_equal_balance & matters["FirstPaymentDate"].notna(),
    ["M_IDX", "CurrentStatus", "FirstPaymentDate", "OpeningBalance", "CurrentBalance"]
].head(10)

,M_IDX,CurrentStatus,FirstPaymentDate,OpeningBalance,CurrentBalance
13,865780,Closed,2025-02-26,210.90,210.90
19,865866,Closed,2025-04-30,179.71,179.71
24,865884,Closed,2025-03-14,190.85,190.85
27,865900,Closed,2025-03-31,280.00,280.00
28,865913,Closed,2025-03-20,1004.76,1004.76
31,865918,Closed,2025-03-07,171.86,171.86
32,865924,Closed,2025-02-26,280.00,280.00
40,865978,Closed,2025-03-14,632.66,632.66
48,866005,Closed,2025-02-28,1484.10,1484.10
54,866033,Closed,2025-05-19,656.03,656.03


**Finding:** 9,381 matters were identified as retracted accounts based on the combination of `CurrentStatus = "Closed"`, `OpeningBalance = CurrentBalance`, and a missing `FirstPaymentDate`. The remaining 134 Closed matters with unchanged balances had a recorded `FirstPaymentDate` and were therefore not classified as retracted accounts.


In [40]:
retracted_accounts = (
    (matters["CurrentStatus"] == "Closed") &
    equal_balance &
    matters["FirstPaymentDate"].isna()
)
retracted_accounts.sum()

np.int64(9381)

In [41]:
matters = matters.loc[~retracted_accounts].copy()

In [42]:
matters.shape

(46798, 12)

In [43]:
retracted_accounts.sum()

np.int64(9381)

In [44]:
(
    (matters["CurrentStatus"] == "Closed") &
    (matters["OpeningBalance"] == matters["CurrentBalance"]) &
    (matters["FirstPaymentDate"].isna())
).sum()

np.int64(0)

In [45]:
matters["CurrentBalance"].isna().sum()

np.int64(939)

In [46]:
matters.loc[
    matters["CurrentBalance"].isna(),
    "FirstPaymentDate"
].isna().value_counts()

FirstPaymentDate
True     937
False      2
Name: count, dtype: int64

In [47]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].isna(),
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].head(10)

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
2623,1196512,Closed,6908.65,NaN,NaT
2624,1196513,Closed,94001.07,NaN,NaT
2625,1196514,Closed,8326.18,NaN,NaT
2626,1196515,Closed,21390.80,NaN,NaT
2627,1196516,Closed,10254.58,NaN,NaT
2628,1196517,Closed,4303.33,NaN,NaT
2629,1196518,Closed,5165.08,NaN,NaT
2630,1196519,Closed,17559.64,NaN,NaT
2631,1196520,Closed,9912.78,NaN,NaT
2632,1196521,Closed,5874.04,NaN,NaT


In [48]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].notna(),
    ["M_IDX", "CurrentStatus", "HandoverDate", "FirstPaymentDate",
     "OpeningBalance", "CurrentBalance", "DateCreated"]
]

,M_IDX,CurrentStatus,HandoverDate,FirstPaymentDate,OpeningBalance,CurrentBalance,DateCreated
5719,1169872,Closed,2025-02-12,2025-02-17,3013.90,NaN,2025-05-23
35130,1490892,Closed,2025-03-14,2025-03-26,1965.88,NaN,2025-05-23


In [49]:
payments.columns

Index(['M_IDX', 'PaymentDate', 'AmountPaid', 'IsPayedAtClient',
       'IsDebitOrderPayment', 'PaymentMethodID', 'PaymentDescription',
       'AgentID'],
      dtype='str')

In [50]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(payments["M_IDX"]).sum()

np.int64(0)

In [51]:
ptp.columns

Index(['M_IDX', 'PTPCreateDate', 'PaymentMethodID', 'PaymentDescription',
       'PaymentFrequencyID', 'PaymentFrequency', 'FirstPaymentAmount',
       'FirstPaymentDate', 'MonthlyPaymentAmount', 'MonthlyPaymentStartDate',
       'DebitAmount', 'CreditAmount', 'ProjectionPaymentDate',
       'ProjectionDescriptionID', 'ProjectionDescription', 'HistoryID'],
      dtype='str')

In [52]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(ptp["M_IDX"]).sum()

np.int64(0)

In [53]:
matters.loc[
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].isna(),
    ["HandoverDate", "DateCreated"]
].value_counts()

HandoverDate  DateCreated
2025-03-04    2025-05-23     914
2025-02-12    2025-05-23       7
2025-02-10    2025-05-23       5
2025-02-14    2025-05-23       4
2025-02-13    2025-05-23       2
2025-03-14    2025-05-23       2
2025-02-05    2025-05-23       1
2025-02-04    2025-05-23       1
2025-03-10    2025-05-23       1
Name: count, dtype: int64

In [54]:
two_missing_balance = (
    matters["CurrentBalance"].isna() &
    matters["FirstPaymentDate"].notna()
)

matters.loc[two_missing_balance, "M_IDX"].isin(payments["M_IDX"]).sum()

np.int64(2)

In [55]:
two_midx = matters.loc[two_missing_balance, "M_IDX"]

payments[payments["M_IDX"].isin(two_midx)]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
1654,1169872,2025-02-18,3423.71,0,0,1,Direct Deposit,77502
3763,1490892,2025-03-27,400.00,1,0,1,Direct Deposit,-1
3764,1490892,2025-03-31,1565.88,1,0,1,Direct Deposit,-1
3765,1490892,2025-03-31,321.10,0,0,1,Direct Deposit,-1


In [56]:
payments[payments["M_IDX"].isin(two_midx)].groupby("M_IDX")["AmountPaid"].sum()

M_IDX
1169872    3423.71
1490892    2286.98
Name: AmountPaid, dtype: float64

In [57]:
enriched_combined.columns[enriched_combined.columns.str.contains("Balance", case=False)]

Index(['ML1_MaxOpeningBalance', 'ML1_TotalOpeningBalances',
       'ML1_DebtRecoveryBalanceEscalation',
       'ML1_SingleCreditFacitlityBalanceEscalation',
       'ML1_GarageBalanceEscalation', 'ML1_LifeInsuranceBalanceEscalation',
       'ML1_OnemonthpersonalLoanBalanceEscalation',
       'ML1_SecuredPension_PolicyBackedLendingBalanceEscalation',
       'ML1_OpenLimitlessBalanceEscalation',
       'ML1_PersonalLoanBalanceEscalation',
       'ML1_StudentLoansBalanceEscalation', 'ML1_UtilityBalanceEscalation',
       'ML1_OverdraftBalanceEscalation', 'ML1_RentalsAssetBalanceEscalation',
       'ML1_RentalsPropertyBalanceEscalation',
       'ML1_RevolvingCreditNonStoreBalanceEscalation',
       'ML1_ShortTermInsuranceBalanceEscalation',
       'ML1_InstallmentBalanceEscalation',
       'ML1_VehicleFinanceBalanceEscalation',
       'ML1_OpenServicesBalanceEscalation', 'ML1_HomeLoanBalanceEscalation',
       'ML1_RevolvingCredit_StoreCardsBalanceEscalation',
       'ML1_CreditCardBalanceE

In [58]:
"M_IDX" in enriched_combined.columns

True

In [59]:
matters.loc[
    matters["CurrentBalance"].isna() & matters["FirstPaymentDate"].isna(),
    "M_IDX"
].isin(enriched_combined["M_IDX"]).sum()

np.int64(929)

**Finding:** After removing the 9,381 confirmed retracted accounts, 939 matters remained with a missing `CurrentBalance`. Of these, 937 also had a missing `FirstPaymentDate`, while 2 had a recorded `FirstPaymentDate` and corresponding payment records. The 937 matters were all `Closed`, had no matching payment or PTP records, and showed a similar date pattern. However, the available evidence was insufficient to classify them definitively as retracted accounts. They will therefore be retained at this stage and revisited after the datasets have been integrated and the eligibility criteria for the modelling population have been established.



#### 4.4 Negative Current Balances

Negative `CurrentBalance` values may indicate that a matter has been overpaid. Their relationship with the current status of the matter is investigated before deciding whether these records should be excluded or treated differently.


In [60]:
matters[matters["CurrentBalance"] < 0]["CurrentStatus"].value_counts()

CurrentStatus
Closed                 95
Broken PTP             13
Follow-up PTP          10
On Hold                 9
Attempting PTP          8
New Instruction         3
In Progress             2
Request for Closure     1
Name: count, dtype: int64

In [61]:
matters.loc[
    matters["CurrentBalance"] < 0,
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].head(20)

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
3,865730,Closed,1539.77,-300.00,2025-03-10
35,865933,Closed,4840.40,-0.40,2025-02-19
42,865982,Closed,1659.24,-0.76,2025-02-17
47,866003,Closed,1318.56,-564.38,2025-02-28
53,866028,Closed,402.98,-0.02,2025-03-18
302,824164,Closed,152.93,-7.07,2025-05-16
397,866167,Closed,152.41,-0.59,2025-04-07
402,866339,Closed,809.81,-0.19,2025-03-17
413,866372,Closed,352.32,-1788.89,2025-04-03
425,867065,Closed,656.32,-3.68,2025-03-27


In [62]:
matters["M_IDX"].isna().sum()

np.int64(0)

In [63]:
negative_balance = matters["CurrentBalance"] < 0
negative_balance.sum()

np.int64(141)

In [64]:
matters.loc[
    (matters["CurrentBalance"] < 0) &
    (matters["CurrentStatus"] != "Closed"),
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
].sort_values("CurrentBalance")

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
9684,1169866,Broken PTP,2234.02,-20015.30,2025-02-27
53230,1055658,Follow-up PTP,12268.24,-8173.01,2025-04-16
10593,1238265,Broken PTP,1147.59,-1147.59,2025-04-30
5154,1200393,On Hold,4890.20,-1021.27,2025-03-25
34913,1656284,Broken PTP,313.99,-1000.00,2025-05-10
13126,1520039,Attempting PTP,542.20,-957.80,NaT
4543,1170155,Attempting PTP,1717.97,-943.01,2025-03-01
2406,1042753,Attempting PTP,8300.64,-921.83,2025-02-18
4203,1193861,Attempting PTP,13950.40,-851.92,NaT
53243,1169108,Follow-up PTP,1535.66,-831.39,2025-03-06


**Finding:** 141 matters had a negative `CurrentBalance`. Although 46 of these matters were not recorded as `Closed` in the source data, the negative balances indicate that the amount collected exceeded the remaining balance. Following the business rule used in the original project, these matters are treated as financially settled and their `CurrentStatus` is therefore set to `Closed` in the cleaned dataset.


In [65]:
matters.loc[matters["CurrentBalance"] < 0, "CurrentStatus"] = "Closed"

In [66]:
matters.loc[
    matters["CurrentBalance"] < 0,
    "CurrentStatus"
].value_counts()

CurrentStatus
Closed    141
Name: count, dtype: int64

In [67]:
matters["ActivationPeriod"].describe()

count    3771.000000
mean        0.720764
std         0.770642
min         0.000000
25%         0.000000
50%         1.000000
75%         1.000000
max         3.000000
Name: ActivationPeriod, dtype: float64

In [68]:
matters["ActivationPeriod"].value_counts(dropna=False).sort_index()

ActivationPeriod
0.0     1699
1.0     1520
2.0      458
3.0       94
NaN    43027
Name: count, dtype: int64

In [69]:
matters.loc[
    matters["ActivationPeriod"].notna() &
    matters["FirstPaymentDate"].notna(),
    ["HandoverDate", "FirstPaymentDate", "ActivationPeriod"]
].head(10)

,HandoverDate,FirstPaymentDate,ActivationPeriod
0,2025-02-11,2025-02-27,0.0
1,2025-02-11,2025-02-19,0.0
2,2025-02-11,2025-02-20,0.0
3,2025-02-12,2025-03-10,1.0
4,2025-02-12,2025-02-28,0.0
5,2025-02-12,2025-02-24,0.0
6,2025-02-12,2025-02-25,0.0
7,2025-02-12,2025-02-28,0.0
8,2025-02-12,2025-03-14,1.0
9,2025-02-12,2025-03-11,1.0


In [70]:
calculated_activation = (
    (matters["FirstPaymentDate"].dt.year - matters["HandoverDate"].dt.year) * 12
    + (matters["FirstPaymentDate"].dt.month - matters["HandoverDate"].dt.month)
)

activation_check = (
    calculated_activation == matters["ActivationPeriod"]
)

activation_check[matters["ActivationPeriod"].notna()].value_counts()

True    3771
Name: count, dtype: int64

**Finding:** `ActivationPeriod` represents the number of calendar months between `HandoverDate` and `FirstPaymentDate`. All 3,771 non-missing `ActivationPeriod` values were validated against these two dates and matched exactly. Missing values were retained because they indicate that a `FirstPaymentDate` is not available, rather than representing an activation period of zero.


In [71]:
matters.loc[
    matters["FirstPaymentDate"].notna() &
    (matters["FirstPaymentDate"] < matters["HandoverDate"]),
    ["M_IDX", "HandoverDate", "FirstPaymentDate", "ActivationPeriod"]
]

,M_IDX,HandoverDate,FirstPaymentDate,ActivationPeriod


In [72]:
matters["CurrentStatus"].value_counts()

CurrentStatus
Attempting PTP         26466
Closed                  6547
New Instruction         6338
Payment Arrangement     3633
Broken PTP              2804
Re-opened                292
Authentication           271
In Progress              230
On Hold                  151
Follow-up PTP             60
Request for Closure        5
Dispute                    1
Name: count, dtype: int64

In [73]:
activation_can_be_calculated = (
    matters["ActivationPeriod"].isna()
    & matters["HandoverDate"].notna()
    & matters["FirstPaymentDate"].notna()
)

activation_can_be_calculated.sum()

np.int64(0)

**Finding:** `ActivationPeriod` represents the number of calendar months between `HandoverDate` and `FirstPaymentDate`. All 3,771 non-missing `ActivationPeriod` values were validated against these two dates and matched exactly. No records had a missing `ActivationPeriod` while both `HandoverDate` and `FirstPaymentDate` were available, so there were no additional values that could be reliably calculated. Missing values were therefore retained because the required date information was not available.


In [74]:
matters[["HandoverDate", "FirstPaymentDate", "DateCreated"]].agg(["min", "max"])

,HandoverDate,FirstPaymentDate,DateCreated
min,2025-02-03,2025-02-03,2025-05-23
max,2025-05-22,2025-05-22,2025-05-23


In [75]:
matters.loc[
    (matters["HandoverDate"] > matters["DateCreated"]) |
    (matters["FirstPaymentDate"] > matters["DateCreated"]),
    ["M_IDX", "HandoverDate", "FirstPaymentDate", "DateCreated"]
]

,M_IDX,HandoverDate,FirstPaymentDate,DateCreated


### 4.6 Matters Cleaning Completion

The `Matters` dataset has been reviewed for duplicate records, missing identifiers, balance inconsistencies, retracted accounts, negative balances, and date-related inconsistencies. Confirmed retracted accounts were removed, negative-balance matters were classified as `Closed`, and `ActivationPeriod` was validated against its underlying dates.

Records with missing `CurrentBalance` were retained for further consideration during dataset integration and definition of the modelling population. No additional date inconsistencies were identified.

The resulting dataset will be carried forward for integration with the remaining source datasets.


### 4.7 Matters Cleaning Completion

The `Matters` dataset has been cleaned and validated based on the identified data-quality and business-rule requirements. Duplicate matter identifiers were not found, confirmed retracted accounts were removed, negative `CurrentBalance` values were treated as financially settled matters and classified as `Closed`, and `ActivationPeriod` was validated against the underlying dates.

Records with missing `CurrentBalance` were retained because there was insufficient evidence to classify them as invalid at this stage. They will be revisited when the datasets are integrated and the modelling population is defined.

The cleaned `Matters` dataset will now be saved to the `data/processed/` directory. The original raw dataset in `data/raw/` remains unchanged so that the cleaning process can be reproduced.

In [76]:
matters.shape

(46798, 12)

#### Saving the Cleaned Dataset

The cleaned `Matters` dataset is saved as an Excel file in `data/processed/`. This creates a reproducible output of the cleaning process while preserving the original raw dataset.

In [77]:
matters.to_excel("../data/processed/Matters_cleaned.xlsx", index=False)

In [78]:
pd.read_excel("../data/processed/Matters_cleaned.xlsx").shape

(46798, 12)

#### Saved Dataset Verification

The saved `Matters` dataset is reloaded and compared with the cleaned DataFrame to confirm that the exported file preserves the cleaned data.

In [79]:
matters_saved = pd.read_excel("../data/processed/Matters_cleaned.xlsx")

matters.equals(matters_saved)

False

In [80]:
pd.DataFrame({
    "original": matters.dtypes,
    "saved": matters_saved.dtypes
})

,original,saved
M_IDX,int64,int64
HandoverDate,datetime64[us],datetime64[us]
CurrentStatusID,int64,int64
CurrentStatus,str,str
PreviousStatusID,float64,float64
PreviousStatus,str,str
FirstPaymentDate,datetime64[us],datetime64[us]
ActivationPeriod,float64,float64
OpeningBalance,float64,float64
CurrentBalance,float64,float64


In [81]:
import numpy as np

np.allclose(
    matters.select_dtypes(include="number"),
    matters_saved.select_dtypes(include="number"),
    equal_nan=True
)

True

**Finding:** The cleaned `Matters` dataset was successfully exported to `data/processed/Matters_cleaned.xlsx` and reloaded with the expected 46,798 rows and 12 columns. The data types were preserved and the numeric values were confirmed to match within floating-point tolerance. The processed file will be used as the cleaned output while the original raw dataset remains unchanged.

## 5. Payments Cleaning

The `Payments` dataset contains payment transactions associated with individual matters. The dataset will be reviewed for duplicate records, missing or inconsistent values, invalid dates or amounts, and other data-quality issues that could affect the analysis and subsequent integration with the other debt-collection datasets.

Cleaning decisions will be based on the structure of the data, business context, and the requirements of the intended prediction task. The original raw dataset will remain unchanged, while all cleaning will be performed on the working DataFrame.


### 5.1 Initial Data Quality Checks

Before applying any transformations, the structure and completeness of the `Payments` dataset will be assessed. This provides a baseline for identifying missing values, unexpected data types, and potential data-quality issues before cleaning.


In [82]:
payments.isna().sum()

M_IDX                  0
PaymentDate            0
AmountPaid             0
IsPayedAtClient        0
IsDebitOrderPayment    0
PaymentMethodID        0
PaymentDescription     0
AgentID                0
dtype: int64

In [83]:
payments.duplicated().sum()

np.int64(88)

In [84]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 6016 entries, 0 to 6015
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   M_IDX                6016 non-null   int64         
 1   PaymentDate          6016 non-null   datetime64[us]
 2   AmountPaid           6016 non-null   float64       
 3   IsPayedAtClient      6016 non-null   int64         
 4   IsDebitOrderPayment  6016 non-null   int64         
 5   PaymentMethodID      6016 non-null   int64         
 6   PaymentDescription   6016 non-null   str           
 7   AgentID              6016 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(5), str(1)
memory usage: 376.1 KB


In [85]:
payments[payments.duplicated(keep=False)]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
60,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
61,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
62,824697,2025-04-03,470.00,1,0,0,Unspecified,-1
94,825013,2025-05-06,500.00,1,0,0,Unspecified,-1
95,825013,2025-05-06,500.00,1,0,0,Unspecified,-1
...,...,...,...,...,...,...,...,...
5594,1665470,2025-05-06,350.00,1,0,1,Direct Deposit,-1
5938,1793102,2025-05-17,154.05,1,0,0,Unspecified,-1
5939,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1
5940,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1


In [86]:
payments[payments.duplicated(keep=False)]["M_IDX"].value_counts()

M_IDX
868845     7
1660223    7
1663838    7
1205026    6
1660241    6
1660330    6
1660523    6
1660551    6
1662294    6
1665422    6
1665470    6
954855     4
1400821    4
1660489    4
1665152    4
1793102    4
824697     3
908344     3
1167111    3
825013     2
866080     2
908338     2
957569     2
972763     2
974298     2
974349     2
989785     2
1167200    2
1169873    2
1169951    2
1170155    2
1171057    2
1176684    2
1197580    2
1200689    2
1231503    2
1238299    2
1378705    2
1520037    2
1556599    2
1648059    2
1649784    2
1650468    2
1656338    2
1660340    2
1660528    2
1660665    2
1662207    2
1662328    2
Name: count, dtype: int64

In [87]:
payments[payments["M_IDX"] == 868845]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
440,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
441,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
442,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
443,868845,2025-03-07,581.85,1,0,0,Unspecified,76283
444,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
445,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
446,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283


In [88]:
payments[payments["M_IDX"] == 868845].index

RangeIndex(start=440, stop=447, step=1)

In [89]:
payments.drop_duplicates().shape

(5928, 8)

### 5.2 Duplicate Payment Records

Exact duplicate rows were identified by comparing all fields in the `Payments` dataset. Because a matter can legitimately have multiple payment transactions, `M_IDX` was not used as a duplicate identifier.

A total of 88 exact duplicate rows were identified. After removing these duplicates, the dataset contains 5,928 rows across 8 columns.

These records are considered redundant because they contain identical information across all payment fields. Exact duplicate rows will therefore be removed from the working dataset.

In [90]:
payments = payments.drop_duplicates()

In [91]:
payments.shape

(5928, 8)

In [92]:
payments.duplicated().sum()

np.int64(0)

### 5.3 Duplicate Removal

The 88 exact duplicate payment records were removed from the working `Payments` dataset. The resulting dataset contains 5,928 rows and 8 columns. A subsequent duplicate check confirmed that no exact duplicate rows remain.

The original raw `Payments` dataset was not modified.


In [93]:
payments = payments.drop_duplicates()

In [94]:
payments.duplicated().sum()

np.int64(0)

### 5.4 Payment Amount Checks

The `AmountPaid` field represents the monetary value associated with each payment transaction. The distribution of payment amounts will be examined to identify negative values, zero values, unusually large payments, and other potentially anomalous values.

These values will be assessed in the context of the payment data and business rules before any records are modified or excluded.


In [95]:
payments["AmountPaid"].describe()

count     5928.000000
mean       699.138548
std       1993.526180
min     -12000.000000
25%        160.000000
50%        300.000000
75%        772.072500
max      60664.000000
Name: AmountPaid, dtype: float64

In [96]:
payments[payments["AmountPaid"] < 0]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
354,868079,2025-05-03,-300.00,1,0,0,Unspecified,-1
444,868845,2025-03-07,-581.85,1,0,0,Unspecified,76283
545,893887,2025-05-21,-326.03,0,0,1,Direct Deposit,74811
699,897402,2025-05-05,-500.00,0,1,29,DebiCheck Batched,73872
960,953986,2025-03-27,-1514.94,0,1,39,RMS,77547
...,...,...,...,...,...,...,...,...
5680,1667153,2025-05-06,-222.22,0,0,1,Direct Deposit,-1
5684,1667179,2025-05-06,-250.00,0,0,1,Direct Deposit,-1
5806,1713470,2025-05-10,-490.18,0,1,39,RMS,77818
5939,1793102,2025-05-17,-154.05,1,0,0,Unspecified,-1


In [97]:
negative_amounts = payments.loc[payments["AmountPaid"] < 0, "AmountPaid"].abs().unique()

negative_amounts

array([  300.  ,   581.85,   326.03,   500.  ,  1514.94,   704.  ,
         750.  ,   550.  ,  1000.  ,  1662.  ,   350.  ,   610.  ,
         600.  ,   700.  ,  2755.17,   430.  ,  1100.  ,   450.  ,
         400.  ,  1376.  ,   436.59,   200.  ,   800.  ,   978.  ,
         150.  ,  1500.  ,  2000.  ,   940.  , 12000.  ,  1388.  ,
        2500.  ,  2726.  ,  3122.74,   620.  ,   100.  ,  1300.  ,
         250.  ,    82.09,    38.88,    26.21, 10205.55,  2774.39,
         745.77,   125.  ,  1352.36,  7123.05,  1090.29,  1184.18,
         915.85,  1430.54,  4287.77,   705.68,  1231.75,   320.  ,
         667.51,  2040.96,  1200.  ,   918.06,  2163.62,   591.19,
         532.99,   277.28,   671.96,  3577.96,  1414.15,  2323.54,
        5052.07,   760.95,   205.08,  2359.  ,  2357.4 ,  3467.48,
        1530.41,  1635.61,  1516.1 ,  3142.71,   406.46,  1628.35,
        1019.53,  2186.06,   369.  ,  1135.43,   195.  ,   915.23,
        2698.28,  2147.19,   425.64,  1623.33,  7144.24,  1049

In [98]:
negative_payments = payments[payments["AmountPaid"] < 0].copy()

negative_payments["matching_positive"] = negative_payments.apply(
    lambda row: (
        (payments["M_IDX"] == row["M_IDX"]) &
        (payments["AmountPaid"] == abs(row["AmountPaid"]))
    ).any(),
    axis=1
)

negative_payments["matching_positive"].value_counts()

matching_positive
True     449
False      3
Name: count, dtype: int64

In [99]:
negative_payments[negative_payments["matching_positive"] == False]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID,matching_positive
2673,1388320,2025-03-11,-82.09,0,0,0,Unspecified,-1,False
2754,1390367,2025-03-11,-38.88,0,0,0,Unspecified,76714,False
3001,1402386,2025-02-22,-26.21,0,0,0,Unspecified,77249,False


In [100]:
payments[
    payments["M_IDX"].isin([1388320, 1390367, 1402386])
].sort_values(["M_IDX", "PaymentDate"])

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
2673,1388320,2025-03-11,-82.09,0,0,0,Unspecified,-1
2674,1388320,2025-03-11,20944.97,0,0,1,Direct Deposit,-1
2753,1390367,2025-03-11,2870.00,0,0,1,Direct Deposit,76714
2754,1390367,2025-03-11,-38.88,0,0,0,Unspecified,76714
3000,1402386,2025-02-22,1470.00,0,0,1,Direct Deposit,77249
3001,1402386,2025-02-22,-26.21,0,0,0,Unspecified,77249


**Finding:** Negative payment amounts were investigated rather than automatically treated as invalid records. Of the 452 negative payment transactions, 449 had an exact matching positive payment for the same matter and amount, indicating that they are likely associated with payment reversals or adjustments. The remaining 3 negative transactions did not have an exact amount match, but each occurred on the same date as a positive payment for the same matter and was associated with the same agent. These records were therefore retained, as there was insufficient evidence to classify them as erroneous. Removing negative transactions could also distort the actual amount collected.

### 5.5 Zero Payment Amounts

A payment transaction with an `AmountPaid` of zero does not represent a monetary payment. These records will therefore be investigated to determine whether they are legitimate transaction records, administrative entries, or potential data-quality issues before deciding whether they should be removed.

In [101]:
payments[payments["AmountPaid"] == 0]

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID


**Finding:** No payment transactions had an `AmountPaid` value of zero. No records were removed based on zero payment amounts.

### 5.6 Unusually Large Payment Amounts

The `AmountPaid` distribution contains a small number of relatively large transactions. These records will be investigated to determine whether they represent legitimate payments or potential data-quality issues. Large payment amounts will not be removed solely because they are unusual, as debt amounts can vary substantially between matters.

In [102]:
payments.nlargest(10, "AmountPaid")

,M_IDX,PaymentDate,AmountPaid,IsPayedAtClient,IsDebitOrderPayment,PaymentMethodID,PaymentDescription,AgentID
4124,1520319,2025-05-12,60664.00,0,0,1,Direct Deposit,76713
4287,1556524,2025-04-27,55635.30,1,0,0,Unspecified,-1
153,862317,2025-02-15,39054.80,1,0,0,Unspecified,-1
4754,1660209,2025-05-07,33445.06,0,0,1,Direct Deposit,75892
64,824701,2025-03-07,25791.70,1,0,0,Unspecified,-1
4410,1627745,2025-04-26,24575.30,1,0,0,Unspecified,-1
1648,1169866,2025-04-08,22535.00,1,0,1,Direct Deposit,-1
1259,1002772,2025-02-27,21306.60,1,0,1,Direct Deposit,-1
2674,1388320,2025-03-11,20944.97,0,0,1,Direct Deposit,-1
4345,1566778,2025-04-25,17865.50,0,0,1,Direct Deposit,77554


In [103]:
large_payments = payments.nlargest(10, "AmountPaid")

large_payments.merge(
    matters[["M_IDX", "OpeningBalance", "CurrentBalance", "CurrentStatus"]],
    on="M_IDX",
    how="left"
)[[
    "M_IDX",
    "AmountPaid",
    "OpeningBalance",
    "CurrentBalance",
    "CurrentStatus"
]]

,M_IDX,AmountPaid,OpeningBalance,CurrentBalance,CurrentStatus
0,1520319,60664.00,74978.80,14438.10,Closed
1,1556524,55635.30,1245.89,245.89,Attempting PTP
2,862317,39054.80,2503.67,3.67,Closed
3,1660209,33445.06,32886.16,48.35,Broken PTP
4,824701,25791.70,6335.03,-25475.16,Closed
5,1627745,24575.30,1884.07,-17000.00,Closed
6,1169866,22535.00,2234.02,-20015.30,Closed
7,1002772,21306.60,44255.14,21323.89,Payment Arrangement
8,1388320,20944.97,25394.54,5017.22,Closed
9,1566778,17865.50,21542.62,0.00,Closed


**Finding:** The largest payment transactions were reviewed against the corresponding matter balances. Several large payments exceeded the original `OpeningBalance` and resulted in negative `CurrentBalance` values, while others were within the original balance. These values are therefore consistent with legitimate payment activity, including overpayments, rather than indicating obvious data-entry errors. No payment records were removed based on payment size.

### 5.7 Payment Date Checks

Payment dates are reviewed to identify transactions that fall outside the expected data-collection period. The overall date range will first be examined, followed by checks against the corresponding matter's `HandoverDate` and the project pull date.

In [104]:
payments["PaymentDate"].agg(["min", "max"])

min   2025-02-06
max   2025-05-22
Name: PaymentDate, dtype: datetime64[us]

In [105]:
payment_handover_check = payments.merge(
    matters[["M_IDX", "HandoverDate"]],
    on="M_IDX",
    how="left"
)

payment_handover_check.loc[
    payment_handover_check["PaymentDate"] < payment_handover_check["HandoverDate"],
    ["M_IDX", "PaymentDate", "HandoverDate"]
]

,M_IDX,PaymentDate,HandoverDate


**Finding:** No payment transactions occurred before the corresponding matter's `HandoverDate`. The payment dates are therefore temporally consistent with the matter handover dates.

In [106]:
payments.loc[
    payments["PaymentDate"] > pd.Timestamp("2025-05-23"),
    ["M_IDX", "PaymentDate", "AmountPaid"]
]

,M_IDX,PaymentDate,AmountPaid


**Finding:** No payment transactions occurred after the project pull date of 23 May 2025. The payment data therefore falls within the expected observation period.

In [107]:
payments["M_IDX"].nunique(), len(payments)

(3786, 5928)

In [108]:
payments.loc[
    ~payments["M_IDX"].isin(matters["M_IDX"]),
    "M_IDX"
].nunique()

0

**Finding:** All payment transactions have a corresponding `M_IDX` in the cleaned `Matters` dataset. No unmatched payment records were identified, confirming that the `Payments` dataset can be linked to `Matters` using `M_IDX`.

### 5.8 Payments Cleaning Completion

The `Payments` dataset has been reviewed for missing values, duplicate records, payment amount anomalies, and temporal inconsistencies. Exact duplicate rows were removed, while negative and unusually large payment amounts were investigated and retained because they were consistent with legitimate payment activity.

No zero-value payments were identified. Payment dates were within the expected project period, no payments occurred before the corresponding matter's `HandoverDate`, and no payments occurred after the 23 May 2025 project pull date. All payment records also had a corresponding `M_IDX` in the cleaned `Matters` dataset.

The cleaned `Payments` dataset will now be saved to the `data/processed/` directory. The original raw dataset remains unchanged.

#### Final Dataset Check

Before saving the cleaned dataset, the final number of rows and columns is checked to confirm that the expected cleaning steps have been applied.

In [109]:
payments.shape

(5928, 8)

#### Saving the Cleaned Dataset

The cleaned `Payments` dataset is saved as an Excel file in `data/processed/`. This creates a reproducible output of the cleaning process while preserving the original raw dataset.

In [110]:
payments.to_excel("../data/processed/Payments_cleaned.xlsx", index=False)

In [111]:
payments_saved = pd.read_excel("../data/processed/Payments_cleaned.xlsx")

payments_saved.shape

(5928, 8)

#### Saved Dataset Verification

The saved `Payments` dataset is reloaded and its structure is compared with the cleaned DataFrame to confirm that the exported file preserves the expected data types and values.

In [112]:
pd.DataFrame({
    "original": payments.dtypes,
    "saved": payments_saved.dtypes
})

,original,saved
M_IDX,int64,int64
PaymentDate,datetime64[us],datetime64[us]
AmountPaid,float64,float64
IsPayedAtClient,int64,int64
IsDebitOrderPayment,int64,int64
PaymentMethodID,int64,int64
PaymentDescription,str,str
AgentID,int64,int64


In [113]:
np.allclose(
    payments.select_dtypes(include="number"),
    payments_saved.select_dtypes(include="number"),
    equal_nan=True
)

True

**Finding:** The cleaned `Payments` dataset was successfully exported to `data/processed/Payments_cleaned.xlsx` and reloaded with the expected 5,928 rows and 8 columns. The data types were preserved and the numeric values were confirmed to match within floating-point tolerance. The processed file will be used as the cleaned output while the original raw dataset remains unchanged.

## 6. PTP Cleaning

The `PTP` dataset contains promise-to-pay records associated with individual matters. The dataset will be reviewed for duplicate records, missing or inconsistent values, invalid dates or amounts, and other data-quality issues that could affect the analysis and subsequent integration with the other debt-collection datasets.

Cleaning decisions will be based on the structure of the data, business context, and the requirements of the intended prediction task. The original raw dataset will remain unchanged, while all cleaning will be performed on the working DataFrame.

### 6.1 Initial Data Quality Checks

In [114]:
ptp.isna().sum()

M_IDX                         0
PTPCreateDate                 0
PaymentMethodID               0
PaymentDescription            0
PaymentFrequencyID            0
PaymentFrequency           1835
FirstPaymentAmount          161
FirstPaymentDate           6174
MonthlyPaymentAmount          0
MonthlyPaymentStartDate    1834
DebitAmount                9391
CreditAmount                  0
ProjectionPaymentDate         0
ProjectionDescriptionID       0
ProjectionDescription         0
HistoryID                     0
dtype: int64

**Finding:** Missing values were identified in five PTP fields: `PaymentFrequency`, `FirstPaymentAmount`, `FirstPaymentDate`, `MonthlyPaymentStartDate`, and `DebitAmount`. The remaining fields contain no missing values. The identified missing values will be investigated further to determine whether they represent legitimate characteristics of PTP arrangements or data-quality issues.

### 6.2 Missing Values

In [115]:
ptp[ptp["PaymentFrequency"].isna()][["PaymentFrequencyID", "PaymentFrequency"]]

,PaymentFrequencyID,PaymentFrequency
0,5,NaN
1,5,NaN
2,5,NaN
3,5,NaN
6,5,NaN
...,...,...
9377,5,NaN
9378,5,NaN
9383,5,NaN
9390,5,NaN


In [116]:
ptp.groupby("PaymentFrequencyID")["PaymentFrequency"].value_counts(dropna=False)

PaymentFrequencyID  PaymentFrequency
1                   Weekly               129
2                   Fortnightly           59
3                   Monthly             7379
5                   NaN                 1835
Name: count, dtype: int64

**Finding:** `PaymentFrequency` is missing for 1,835 records. All records with a missing `PaymentFrequency` have `PaymentFrequencyID = 5`, while IDs 1, 2, and 3 consistently correspond to Weekly, Fortnightly, and Monthly payment frequencies respectively. This indicates that the missing `PaymentFrequency` values are associated with a specific category rather than occurring randomly. Because the meaning of `PaymentFrequencyID = 5` has not yet been established, the missing values will be retained rather than imputed.

In [117]:
ptp[ptp["FirstPaymentAmount"].isna()][
    ["FirstPaymentDate", "FirstPaymentAmount",
     "MonthlyPaymentAmount", "MonthlyPaymentStartDate"]
].head(20)

,FirstPaymentDate,FirstPaymentAmount,MonthlyPaymentAmount,MonthlyPaymentStartDate
318,NaT,NaN,600.0,2025-04-16
319,NaT,NaN,600.0,2025-04-16
339,NaT,NaN,500.0,2025-05-20
357,NaT,NaN,500.0,2025-04-29
358,NaT,NaN,500.0,2025-04-29
540,NaT,NaN,172.0,2025-03-31
619,NaT,NaN,3000.0,2025-05-02
631,NaT,NaN,172.0,2025-03-31
632,NaT,NaN,172.0,2025-03-31
706,NaT,NaN,1500.0,2025-04-25


In [118]:
ptp[ptp["FirstPaymentAmount"].isna()]["FirstPaymentDate"].isna().sum()

np.int64(161)

**Finding:** The 161 records with missing `FirstPaymentAmount` were investigated further. All 161 records also have a missing `FirstPaymentDate`, while the records can contain populated `MonthlyPaymentAmount` and `MonthlyPaymentStartDate` values. This indicates that the missing first-payment fields do not necessarily represent invalid PTP records, as a monthly payment arrangement may still be recorded. The records will therefore be retained without imputing the missing first-payment values.

In [119]:
ptp[ptp["FirstPaymentDate"].isna()]["FirstPaymentAmount"].notna().sum()

np.int64(6013)

In [120]:
ptp[ptp["FirstPaymentDate"].isna()]["MonthlyPaymentStartDate"].notna().sum()

np.int64(6174)

**Finding:** `FirstPaymentDate` is missing for 6,174 PTP records. Further investigation showed that all 6,174 records have a populated `MonthlyPaymentStartDate`, while 6,013 have a recorded `FirstPaymentAmount` and 161 have both `FirstPaymentAmount` and `FirstPaymentDate` missing. Since the PTP dataset records promised payment arrangements rather than actual payment transactions, the absence of `FirstPaymentDate` does not by itself indicate an invalid record. No records were removed or imputed based on this missingness.

In [121]:
ptp[ptp["MonthlyPaymentStartDate"].isna()][
    ["PaymentFrequencyID", "PaymentFrequency", "MonthlyPaymentAmount"]
].head(20)

,PaymentFrequencyID,PaymentFrequency,MonthlyPaymentAmount
0,5,NaN,0.0
1,5,NaN,0.0
2,5,NaN,0.0
3,5,NaN,0.0
6,5,NaN,0.0
7,5,NaN,0.0
12,5,NaN,0.0
13,5,NaN,0.0
36,5,NaN,0.0
46,5,NaN,0.0


In [122]:
ptp[ptp["MonthlyPaymentStartDate"].isna()]["PaymentFrequencyID"].value_counts()

PaymentFrequencyID
5    1834
Name: count, dtype: int64

In [123]:
ptp[
    (ptp["PaymentFrequencyID"] == 5) &
    (ptp["MonthlyPaymentStartDate"].notna())
][[
    "PaymentFrequencyID",
    "PaymentFrequency",
    "MonthlyPaymentAmount",
    "MonthlyPaymentStartDate"
]]

,PaymentFrequencyID,PaymentFrequency,MonthlyPaymentAmount,MonthlyPaymentStartDate
6043,5,NaN,0.0,2025-06-30


**Finding:** `PaymentFrequency` is missing for 1,835 records, and all of these records have `PaymentFrequencyID = 5`. Payment frequency IDs 1, 2, and 3 consistently correspond to Weekly, Fortnightly, and Monthly respectively. The meaning of ID 5 has not been established from the available data, so the missing `PaymentFrequency` values will be retained rather than imputed.

**Finding:** `MonthlyPaymentStartDate` is missing for 1,834 records, and all of these records have `PaymentFrequencyID = 5`. These records also have a `MonthlyPaymentAmount` of 0.0 based on the investigated records. The remaining `PaymentFrequencyID = 5` record has a `MonthlyPaymentStartDate` of 30 June 2025 but a `MonthlyPaymentAmount` of 0.0. The missing dates are therefore treated as part of the PTP arrangement structure rather than automatically classified as invalid values. No records were removed or imputed based on this field.

In [124]:
ptp[ptp["DebitAmount"].notna()]["DebitAmount"].value_counts()

DebitAmount
0.0    11
Name: count, dtype: int64

**Finding:** `DebitAmount` is missing for 9,391 of the 9,402 PTP records. Only 11 records contain a non-missing value, and all 11 values are 0.0. The field therefore contains very little information in the current dataset. `DebitAmount` will be retained during cleaning and its usefulness will be assessed during subsequent feature selection and modelling.

### 6.3 Duplicate Records

In [125]:
ptp.duplicated().sum()

np.int64(0)

**Finding:** An exact duplicate check was performed across all columns in the `PTP` dataset. No exact duplicate records were identified, so no PTP records were removed on the basis of duplication.

### 6.4 Date Validation

The date fields in the `PTP` dataset will be reviewed to identify invalid, inconsistent, or unexpected dates. Because some PTP dates represent scheduled or projected payments, dates occurring after the 23 May 2025 project pull date will not automatically be treated as invalid. Date validation will therefore consider the purpose of each date field and its relationship to the project timeline.


In [126]:
ptp[[
    "PTPCreateDate",
    "FirstPaymentDate",
    "MonthlyPaymentStartDate",
    "ProjectionPaymentDate"
]].agg(["min", "max"])

,PTPCreateDate,FirstPaymentDate,MonthlyPaymentStartDate,ProjectionPaymentDate
min,2025-02-06,2025-02-08,2025-02-17,2025-02-08
max,2025-05-23,2025-06-01,2025-06-30,2025-05-31


In [127]:
ptp[ptp["PTPCreateDate"] > pd.Timestamp("2025-05-23")][
    ["M_IDX", "PTPCreateDate"]
]

,M_IDX,PTPCreateDate


**Finding:** All `PTPCreateDate` values occur on or before the 23 May 2025 project pull date. No PTP records were identified with a creation date after the project pull date, so no records were removed based on `PTPCreateDate`.


In [128]:
ptp[
    ptp["FirstPaymentDate"].notna() &
    (ptp["FirstPaymentDate"] < ptp["PTPCreateDate"])
][
    ["M_IDX", "PTPCreateDate", "FirstPaymentDate"]
]

,M_IDX,PTPCreateDate,FirstPaymentDate


**Finding:** No records were identified where `FirstPaymentDate` occurred before `PTPCreateDate`. This is consistent with the expected chronology of a PTP arrangement, where the arrangement is created before the promised first payment date. No records were removed based on this check.


In [129]:
ptp[
    ptp["MonthlyPaymentStartDate"].notna() &
    (ptp["MonthlyPaymentStartDate"] < ptp["PTPCreateDate"])
][
    ["M_IDX", "PTPCreateDate", "MonthlyPaymentStartDate"]
]

,M_IDX,PTPCreateDate,MonthlyPaymentStartDate


**Finding:** No records were identified where `MonthlyPaymentStartDate` occurred before `PTPCreateDate`. This is consistent with the expected chronology of a PTP arrangement, where the payment schedule should begin on or after the arrangement creation date. No records were removed based on this check.


In [130]:
ptp[
    ptp["ProjectionPaymentDate"] < ptp["PTPCreateDate"]
][
    ["M_IDX", "PTPCreateDate", "ProjectionPaymentDate"]
]

,M_IDX,PTPCreateDate,ProjectionPaymentDate
547,1469948,2025-04-06,2025-04-05
646,1469185,2025-04-06,2025-04-05
1697,1661850,2025-04-27,2025-04-26


In [131]:
ptp[
    ptp["M_IDX"].isin([1469948, 1469185, 1661850])
].T

,547,548,549,646,1697
M_IDX,1469948,1469948,1469948,1469185,1661850
PTPCreateDate,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-27 00:00:00
PaymentMethodID,1,1,1,1,11
PaymentDescription,Direct Deposit,Direct Deposit,Direct Deposit,Direct Deposit,Pay@
PaymentFrequencyID,3,3,3,3,1
PaymentFrequency,Monthly,Monthly,Monthly,Monthly,Weekly
FirstPaymentAmount,100.0,100.0,100.0,3027.14,290.0
FirstPaymentDate,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-06 00:00:00,2025-04-27 00:00:00
MonthlyPaymentAmount,100.0,100.0,100.0,3027.14,290.0
MonthlyPaymentStartDate,2025-04-07 00:00:00,2025-04-07 00:00:00,2025-04-07 00:00:00,2025-05-06 00:00:00,2025-04-27 00:00:00


**Finding:** Three records were identified where `ProjectionPaymentDate` occurred one day before `PTPCreateDate`. Inspection of the affected records showed that each discrepancy applied to an initial projected payment, while `FirstPaymentDate` was consistent with `PTPCreateDate` and subsequent projected payment dates followed the expected schedule. The discrepancies were therefore retained as they appear to reflect a system-generated date convention rather than invalid PTP records. No records were removed or date values modified based on this check.


In [132]:
ptp[
    ptp["FirstPaymentDate"].notna() &
    (ptp["FirstPaymentDate"] > pd.Timestamp("2025-05-23"))
][
    ["M_IDX", "PTPCreateDate", "FirstPaymentDate"]
]

,M_IDX,PTPCreateDate,FirstPaymentDate
715,1542190,2025-04-29,2025-05-29
716,1542190,2025-04-29,2025-05-29
720,1498564,2025-05-02,2025-05-24
981,1661943,2025-04-29,2025-05-27
1706,1556552,2025-05-06,2025-05-31
...,...,...,...
9374,1791572,2025-05-23,2025-05-26
9378,1793143,2025-05-23,2025-05-28
9383,1519610,2025-05-19,2025-05-31
9388,1390882,2025-05-22,2025-05-31


In [133]:
ptp[
    ptp["FirstPaymentDate"].notna() &
    (ptp["FirstPaymentDate"] > pd.Timestamp("2025-05-23"))
]["M_IDX"].nunique()

515

**Finding:** A total of 523 PTP records, representing 515 unique matters, have a `FirstPaymentDate` after the 23 May 2025 project pull date. These dates represent promised first-payment dates rather than actual payment transactions, so future dates are valid within the context of a PTP arrangement. No records were removed or dates modified based on this check.


### 6.5 Amount Validation

The monetary fields in the `PTP` dataset will be reviewed for negative, zero, or unusually large values. Potential anomalies will be investigated in the context of PTP arrangements before any records or values are removed or modified.


In [134]:
ptp[[
    "FirstPaymentAmount",
    "MonthlyPaymentAmount",
    "DebitAmount"
]].lt(0).sum()

FirstPaymentAmount      0
MonthlyPaymentAmount    0
DebitAmount             0
dtype: int64

**Finding:** No negative values were identified in `FirstPaymentAmount`, `MonthlyPaymentAmount`, or `DebitAmount`. No records were removed or modified based on negative monetary values.


In [135]:
ptp[[
    "FirstPaymentAmount",
    "MonthlyPaymentAmount",
    "DebitAmount"
]].eq(0).sum()

FirstPaymentAmount      6013
MonthlyPaymentAmount    1835
DebitAmount               11
dtype: int64

In [136]:
ptp[
    ptp["FirstPaymentAmount"] == 0
]["FirstPaymentDate"].isna().sum()

np.int64(6013)

In [137]:
ptp[
    ptp["MonthlyPaymentAmount"] == 0
]["PaymentFrequencyID"].value_counts()

PaymentFrequencyID
5    1835
Name: count, dtype: int64

In [138]:
ptp[
    ptp["DebitAmount"].notna()
][[
    "DebitAmount",
    "PaymentMethodID",
    "PaymentDescription",
    "CreditAmount"
]]

,DebitAmount,PaymentMethodID,PaymentDescription,CreditAmount
4836,0.0,29,DebiCheck Batched,1000.00
4837,0.0,29,DebiCheck Batched,1000.00
4838,0.0,29,DebiCheck Batched,1000.00
5591,0.0,29,DebiCheck Batched,500.00
5592,0.0,29,DebiCheck Batched,500.00
5593,0.0,29,DebiCheck Batched,500.00
5681,0.0,39,RMS,490.18
6767,0.0,1,Direct Deposit,300.00
7148,0.0,29,DebiCheck Batched,222.91
7844,0.0,27,DebiCheck Realtime,160.15


**Finding:** Zero values were identified in all three monetary fields. All 6,013 records with `FirstPaymentAmount = 0` also have a missing `FirstPaymentDate`, indicating that these values are associated with PTP records where a first payment was not specified. All 1,835 records with `MonthlyPaymentAmount = 0` have `PaymentFrequencyID = 5`, the same group for which `PaymentFrequency` is missing. The 11 non-missing `DebitAmount` values are all 0.0, while the corresponding records contain positive `CreditAmount` values across several payment methods. These zero values were therefore retained rather than treated as invalid records.


In [139]:
ptp[[
    "FirstPaymentAmount",
    "MonthlyPaymentAmount",
    "DebitAmount"
]].describe()

,FirstPaymentAmount,MonthlyPaymentAmount,DebitAmount
count,9241.000000,9402.000000,11.0
mean,349.539513,496.435433,0.0
std,1684.565959,1221.142633,0.0
min,0.000000,0.000000,0.0
25%,0.000000,100.000000,0.0
50%,0.000000,250.000000,0.0
75%,222.910000,500.000000,0.0
max,62058.620000,54293.920000,0.0


In [140]:
ptp.nlargest(10, "FirstPaymentAmount")[
    [
        "M_IDX",
        "FirstPaymentAmount",
        "FirstPaymentDate",
        "MonthlyPaymentAmount",
        "PaymentDescription"
    ]
]

,M_IDX,FirstPaymentAmount,FirstPaymentDate,MonthlyPaymentAmount,PaymentDescription
9079,1729756,62058.62,2025-05-05,0.00,Direct Deposit
8715,1520319,60663.19,2025-05-31,0.00,Direct Deposit
4158,1144326,54293.92,2025-03-31,54293.92,Direct Deposit
5500,1713703,49215.89,2025-05-31,0.00,Direct Deposit
8493,1801534,37728.14,2025-05-24,0.00,Direct Deposit
5012,1660209,33522.36,2025-05-13,0.00,Direct Deposit
7566,1809263,26392.28,2025-05-24,0.00,Direct Deposit
5184,957757,24549.32,2025-04-30,0.00,Direct Deposit
8346,1169476,21866.14,2025-04-25,0.00,Direct Deposit
2078,1492106,21433.24,2025-04-28,21433.24,Direct Deposit


In [141]:
ptp.nlargest(10, "FirstPaymentAmount")[
    ["M_IDX", "FirstPaymentAmount"]
].merge(
    matters[["M_IDX", "OpeningBalance", "CurrentBalance"]],
    on="M_IDX",
    how="left"
)

,M_IDX,FirstPaymentAmount,OpeningBalance,CurrentBalance
0,1729756,62058.62,60851.11,60892.51
1,1520319,60663.19,74978.80,14438.10
2,1144326,54293.92,53702.70,55018.53
3,1713703,49215.89,48085.03,48091.93
4,1801534,37728.14,46347.67,46437.37
5,1660209,33522.36,32886.16,48.35
6,1809263,26392.28,32251.16,32282.21
7,957757,24549.32,23901.59,24057.66
8,1169476,21866.14,20502.43,21320.90
9,1492106,21433.24,26061.27,24309.77


**Finding:** The largest `FirstPaymentAmount` values were investigated against the corresponding matter balances. The large promised amounts were generally of a similar magnitude to the associated `OpeningBalance` values, although some exceeded the opening balance slightly. These values are therefore considered plausible PTP amounts rather than clear data-entry errors. No PTP records were removed or amounts modified based on payment size.


In [142]:
ptp.nlargest(10, "MonthlyPaymentAmount")[
    [
        "M_IDX",
        "FirstPaymentAmount",
        "MonthlyPaymentAmount",
        "FirstPaymentDate",
        "PaymentDescription"
    ]
]

,M_IDX,FirstPaymentAmount,MonthlyPaymentAmount,FirstPaymentDate,PaymentDescription
4158,1144326,54293.92,54293.92,2025-03-31,Direct Deposit
8300,1728984,0.00,50000.00,NaT,Direct Deposit
2078,1492106,21433.24,21433.24,2025-04-28,Direct Deposit
1153,1390265,0.00,20500.00,NaT,Debit Order
3836,1713916,0.00,20435.00,NaT,Direct Deposit
609,1166827,0.00,20000.00,NaT,Direct Deposit
610,1166827,0.00,20000.00,NaT,Direct Deposit
7054,1388477,0.00,18352.33,NaT,Direct Deposit
7055,1388477,0.00,18352.33,NaT,Direct Deposit
5928,1731370,0.00,15751.00,NaT,Direct Deposit


**Finding:** The largest `MonthlyPaymentAmount` values were reviewed. The values were associated with PTP arrangements containing substantial promised payment amounts and did not show an obvious data-entry pattern requiring removal. Several records had a zero `FirstPaymentAmount` alongside a positive `MonthlyPaymentAmount`, which is consistent with the earlier finding that the first payment amount may be unspecified while a recurring payment amount is recorded. No PTP records were removed or amounts modified based on the size of `MonthlyPaymentAmount`.


### 6.6 PTP–Matters Link Validation

The `PTP` dataset will be checked against the cleaned `Matters` dataset to confirm that PTP records can be linked to a corresponding matter using `M_IDX`. This validation is important before the datasets are integrated, as PTP records without a matching matter may require further investigation or exclusion during integrati_


In [143]:
ptp.loc[~ptp["M_IDX"].isin(matters["M_IDX"]), "M_IDX"].nunique()

7

In [144]:
ptp.loc[
    ~ptp["M_IDX"].isin(matters["M_IDX"]),
    ["M_IDX", "PTPCreateDate", "PaymentDescription", "FirstPaymentAmount", "MonthlyPaymentAmount"]
].drop_duplicates()

,M_IDX,PTPCreateDate,PaymentDescription,FirstPaymentAmount,MonthlyPaymentAmount
1354,1519547,2025-04-01,Direct Deposit,0.00,219.93
1479,1634274,2025-04-14,DebiCheck Batched,0.00,1110.00
1738,1519497,2025-04-24,Direct Deposit,0.00,217.78
3564,1519543,2025-04-04,Direct Deposit,6166.92,0.00
4443,1708947,2025-05-05,RMS,0.00,1000.00
5147,1708744,2025-05-05,RMS,0.00,500.00
6475,1414982,2025-03-27,Direct Deposit,700.00,0.00


In [145]:
id_missing_ptp = ptp.loc[
    ~ptp["M_IDX"].isin(matters["M_IDX"]),
    "M_IDX"
].unique()

matters_raw = pd.read_excel("../data/raw/Matters.xlsx", sheet_name="Data")

matters_raw.loc[
    matters_raw["M_IDX"].isin(id_missing_ptp),
    ["M_IDX", "CurrentStatus", "OpeningBalance", "CurrentBalance", "FirstPaymentDate"]
]

,M_IDX,CurrentStatus,OpeningBalance,CurrentBalance,FirstPaymentDate
16466,1519497,Closed,435.57,435.57,NaT
16964,1519543,Closed,6166.92,6166.92,NaT
16966,1519547,Closed,219.93,219.93,NaT
21111,1708744,Closed,34771.76,34771.76,NaT
21759,1414982,Closed,18322.68,18322.68,NaT
22562,1708947,Closed,19288.19,19288.19,NaT
28576,1634274,Closed,6045.57,6045.57,NaT


**Finding:** Seven unique `M_IDX` values in the `PTP` dataset do not appear in the cleaned `Matters` dataset. Investigation against the original `Matters` dataset confirmed that all seven records were present in the raw data and had been removed during Matters cleaning because they met the defined retracted-account rule: `CurrentStatus = "Closed"`, `OpeningBalance = CurrentBalance`, and missing `FirstPaymentDate`. The PTP records were therefore not considered invalid or orphaned records. They will be retained in the cleaned `PTP` dataset, with their exclusion from any integrated modelling population handled when the final integration and modelling population are defined.


### 6.7 PTP Cleaning Completion

The `PTP` dataset has been reviewed for missing values, duplicate records, date inconsistencies, and monetary-value anomalies. Missing values were investigated in the context of PTP arrangements, while zero and unusually large payment amounts were reviewed and retained where they were considered plausible. Date checks identified no substantive chronology issues requiring modification, and all PTP records were retained because the identified anomalies were consistent with the structure of the dataset.

The relationship between `PTP` and the cleaned `Matters` dataset was also validated. Seven PTP matters were not present in the cleaned `Matters` dataset because they had been deliberately removed as retracted accounts during Matters cleaning. These records will be retained in the cleaned PTP dataset, with their treatment during dataset integration determined when the final modelling population is defined.

The cleaned `PTP` dataset will now be saved to the `data/processed/` directory. The original raw dataset will remain unchanged.


In [146]:
ptp.shape

(9402, 16)

### 6.8 Save Cleaned PTP Dataset

The cleaned `PTP` dataset has been validated and contains the expected 9,402 rows and 16 columns. The cleaned dataset will now be exported to the `data/processed/` directory. The original raw `PTP` dataset in `data/raw/` will remain unchanged so that the cleaning process can be reproduced.


In [147]:
ptp.to_excel("../data/processed/PTP_cleaned.xlsx", index=False)

### 6.9 Verify Saved PTP Dataset

The exported `PTP` dataset will be reloaded to confirm that the processed file was saved successfully and retains the expected number of rows and columns.


In [148]:
ptp_saved = pd.read_excel("../data/processed/PTP_cleaned.xlsx")

ptp_saved.shape

(9402, 16)

**Finding:** The cleaned `PTP` dataset was successfully exported to `data/processed/PTP_cleaned.xlsx` and reloaded with the expected 9,402 rows and 16 columns. The processed file will be used as the cleaned PTP output while the original raw dataset remains unchanged.


## 7. CallHistory Cleaning

The `CallHistory` dataset contains records of contact attempts and call activity associated with individual matters. The dataset will be reviewed for duplicate records, missing or inconsistent values, invalid dates or identifiers, and other data-quality issues that could affect analysis and subsequent integration with the other debt-collection datasets.

Cleaning decisions will be based on the structure of the data, business context, and the requirements of the intended prediction task. The original raw dataset will remain unchanged, while all cleaning will be performed on the working DataFrame.


### 7.1 Initial Data Quality Checks

The `CallHistory` dataset will first be reviewed for missing values across its fields. Missing values will then be investigated individually to determine whether they represent legitimate characteristics of call records or potential data-quality issues.


In [149]:
call_history.isna().sum()

M_IDX                      0
AgentID                    0
HistoryID                  0
CallDate                   0
MinutesOfCall         226131
HasRPC                347689
CallTypeID             80073
CallType                   0
PTPCreateIndicator    421988
TimeCallStart         221181
TimeCallConfirmRPC    412781
TimeCallEnded         226131
dtype: int64

**Finding:** Missing values were identified in `MinutesOfCall`, `HasRPC`, `CallTypeID`, `PTPCreateIndicator`, `TimeCallStart`, `TimeCallConfirmRPC`, and `TimeCallEnded`. The remaining fields contain no missing values. Because several of the affected fields describe events or outcomes that may not occur for every call record, each missing field will be investigated in the context of the call-history structure before any values are imputed or records are removed.


In [150]:
call_history[call_history["MinutesOfCall"].isna()][
    ["TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]
].isna().sum()

TimeCallStart         221181
TimeCallConfirmRPC    225634
TimeCallEnded         226131
dtype: int64

In [151]:
call_history[
    call_history["MinutesOfCall"].isna() &
    (
        call_history["TimeCallStart"].notna() |
        call_history["TimeCallConfirmRPC"].notna()
    )
][
    [
        "M_IDX",
        "CallDate",
        "CallTypeID",
        "CallType",
        "MinutesOfCall",
        "TimeCallStart",
        "TimeCallConfirmRPC",
        "TimeCallEnded"
    ]
].head(20)

,M_IDX,CallDate,CallTypeID,CallType,MinutesOfCall,TimeCallStart,TimeCallConfirmRPC,TimeCallEnded
5,820363,2025-02-07,21.0,Inbound call,NaN,2025-02-07 08:39:14.047,NaN,NaN
18,817515,2025-02-13,22.0,Outbound call,NaN,2025-02-13 14:37:50.260,NaN,NaN
91,865719,2025-02-14,21.0,Inbound call,NaN,2025-02-14 11:01:17.297,NaN,NaN
94,865776,2025-02-14,21.0,Inbound call,NaN,2025-02-14 11:06:21.037,NaN,NaN
103,862293,2025-02-14,22.0,Outbound call,NaN,2025-02-14 11:12:49.323,2025-02-14 11:16:13.800,NaN
108,862285,2025-02-14,21.0,Inbound call,NaN,2025-02-14 11:21:02.900,NaN,NaN
112,862265,2025-02-14,22.0,Outbound call,NaN,2025-02-14 11:21:41.050,2025-02-14 11:24:06.913,NaN
308,865871,2025-02-14,21.0,Inbound call,NaN,2025-02-14 11:48:20.893,NaN,NaN
310,865741,2025-02-14,21.0,Inbound call,NaN,2025-02-14 13:13:00.547,NaN,NaN
1101,865911,2025-02-18,22.0,Outbound call,NaN,2025-02-18 09:29:52.650,NaN,NaN


**Finding:** Most records with missing `MinutesOfCall` also have missing call timing information, particularly `TimeCallEnded`. A smaller subset has a populated `TimeCallStart` and/or `TimeCallConfirmRPC` but no `TimeCallEnded`, meaning that a reliable call duration cannot be calculated from the available timestamps. `MinutesOfCall` will therefore be retained as missing rather than imputed or calculated from incomplete timing information.


In [152]:
call_history[call_history["HasRPC"].isna()][
    ["CallTypeID", "CallType", "HasRPC"]
].groupby(
    ["CallTypeID", "CallType"],
    dropna=False
).size()

CallTypeID  CallType     
21.0        Inbound call        164
22.0        Outbound call    267452
NaN         Inbound call      80073
dtype: int64

In [153]:
call_history.groupby(
    ["CallTypeID", "CallType"],
    dropna=False
)["HasRPC"].value_counts(dropna=False)

CallTypeID  CallType       HasRPC
21.0        Inbound call   1.0          798
                           NaN          164
                           0.0           26
22.0        Outbound call  NaN       267452
                           0.0        64848
                           1.0        16836
NaN         Inbound call   NaN        80073
Name: count, dtype: int64

In [155]:
call_history[call_history["HasRPC"].isna()][
    ["MinutesOfCall", "TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]
].isna().sum()

MinutesOfCall         222307
TimeCallStart         221179
TimeCallConfirmRPC    347689
TimeCallEnded         222307
dtype: int64

**Finding:** `HasRPC` is missing for 347,689 call records. All records with missing `HasRPC` also have a missing `TimeCallConfirmRPC`, while the majority also have missing call duration, start time, and end time. This indicates that missing `HasRPC` values are primarily associated with call records where no RPC outcome was recorded. Consistent with the structure of the field and the previous cleaning approach, missing `HasRPC` values will be treated as `0` (no recorded Right Party Contact).


In [156]:
call_history["HasRPC"] = call_history["HasRPC"].fillna(0)

In [157]:
call_history["HasRPC"].isna().sum()

np.int64(0)

In [158]:
call_history.groupby(
    "CallType"
)["CallTypeID"].value_counts(dropna=False)

CallType       CallTypeID
Inbound call   NaN            80073
               21.0             988
Outbound call  22.0          349136
Name: count, dtype: int64

**Finding:** `CallTypeID` is missing for 80,073 records, all of which are classified as `Inbound call`. The observed relationship between `CallType` and `CallTypeID` is consistent: inbound calls have `CallTypeID = 21`, while outbound calls have `CallTypeID = 22`, with no conflicting combinations observed. Based on this consistent relationship, the missing `CallTypeID` values can be reliably inferred as `21` for inbound calls.


In [159]:
call_history.loc[
    call_history["CallTypeID"].isna() &
    (call_history["CallType"] == "Inbound call"),
    "CallTypeID"
] = 21

In [160]:
call_history["CallTypeID"].isna().sum()

np.int64(0)

In [161]:
call_history["PTPCreateIndicator"].value_counts(dropna=False)

PTPCreateIndicator
NaN    421988
1.0      8209
Name: count, dtype: int64

In [162]:
ptp_matters = set(ptp["M_IDX"].unique())

call_history["HasPTPRecord"] = call_history["M_IDX"].isin(ptp_matters)

call_history.groupby(
    ["PTPCreateIndicator", "HasPTPRecord"],
    dropna=False
).size()

PTPCreateIndicator  HasPTPRecord
1.0                 False              779
                    True              7430
NaN                 False           339650
                    True             82338
dtype: int64

**Finding:** `PTPCreateIndicator` contains 8,209 records with a value of `1` and 421,988 missing values, with no observed `0` values. Comparison with the `PTP` dataset showed that `PTPCreateIndicator = 1` is strongly associated with the existence of a PTP record for the corresponding matter; however, 82,338 call records with missing `PTPCreateIndicator` belong to matters that have PTP records. This indicates that the indicator likely describes whether a PTP was created during a specific call event rather than whether the matter has ever had a PTP. The missing values will therefore be retained rather than automatically imputed as `0`.


In [163]:
call_history = call_history.drop(columns="HasPTPRecord")

In [164]:
call_history.columns

Index(['M_IDX', 'AgentID', 'HistoryID', 'CallDate', 'MinutesOfCall', 'HasRPC',
       'CallTypeID', 'CallType', 'PTPCreateIndicator', 'TimeCallStart',
       'TimeCallConfirmRPC', 'TimeCallEnded'],
      dtype='str')

### 7.2 Duplicate Records

The `CallHistory` dataset will be checked for duplicate records at both the full-row level and the `HistoryID` level. Exact duplicate rows may represent redundant records, while duplicate `HistoryID` values may indicate that the same call-history event has been recorded more than once. `HistoryID` will therefore be investigated as a potential event-level identifier before any records are removed.


In [165]:
call_history.duplicated().sum()

np.int64(0)

In [166]:
call_history["HistoryID"].duplicated().sum()

np.int64(0)

**Finding:** No exact duplicate records were identified in the `CallHistory` dataset. A separate check of `HistoryID` also found no duplicate identifiers, indicating that each call-history record has a unique `HistoryID`. No CallHistory records will therefore be removed on the basis of duplication.


### 7.3 Date and Time Validation

The date and time fields in the `CallHistory` dataset will be reviewed for unexpected date ranges and chronological inconsistencies. Missing timestamps will not automatically be treated as invalid because call records may not contain complete timing information. Where timestamps are present, their sequence will be checked to ensure that call events follow a logical order.


In [167]:
call_history[[
    "CallDate",
    "TimeCallStart",
    "TimeCallConfirmRPC",
    "TimeCallEnded"
]].agg(["min", "max"])

,CallDate,TimeCallStart,TimeCallConfirmRPC,TimeCallEnded
min,2023-07-04,2025-02-06 08:27:30.053,2025-02-06 11:57:00.150,2025-02-06 08:27:42.980
max,2025-05-23,2025-05-23 14:06:25.667,2025-05-23 14:05:45.480,2025-05-23 14:06:25.227


### 7.3.1 Timestamp Chronology

The chronology of the available call timestamps will be checked to identify records where the recorded call events occur in an unexpected order. Comparisons will only be made where the relevant timestamps are present, as missing timestamps may represent incomplete call records rather than invalid data.


In [168]:
call_history[
    call_history["TimeCallStart"].notna() &
    call_history["TimeCallEnded"].notna() &
    (call_history["TimeCallEnded"] < call_history["TimeCallStart"])
][
    ["M_IDX", "HistoryID", "CallDate", "TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]
].shape

(0, 6)

In [169]:
call_history[
    call_history["TimeCallStart"].notna() &
    call_history["TimeCallConfirmRPC"].notna() &
    (
        (call_history["TimeCallConfirmRPC"] < call_history["TimeCallStart"]) |
        (
            call_history["TimeCallEnded"].notna() &
            (call_history["TimeCallConfirmRPC"] > call_history["TimeCallEnded"])
        )
    )
][
    ["M_IDX", "HistoryID", "CallDate", "TimeCallStart",
     "TimeCallConfirmRPC", "TimeCallEnded"]
].shape

(1, 6)

In [172]:
call_history[
    call_history["TimeCallStart"].notna() &
    call_history["TimeCallConfirmRPC"].notna() &
    (
        (call_history["TimeCallConfirmRPC"] < call_history["TimeCallStart"]) |
        (
            call_history["TimeCallEnded"].notna() &
            (call_history["TimeCallConfirmRPC"] > call_history["TimeCallEnded"])
        )
    )
][
    ["M_IDX", "HistoryID", "CallDate", "CallTypeID", "CallType",
     "MinutesOfCall", "HasRPC", "TimeCallStart",
     "TimeCallConfirmRPC", "TimeCallEnded"]
]

,M_IDX,HistoryID,CallDate,CallTypeID,CallType,MinutesOfCall,HasRPC,TimeCallStart,TimeCallConfirmRPC,TimeCallEnded
65541,1232078,44371226,2025-03-24,21.0,Inbound call,1.0,1.0,2025-03-24 11:46:04.490,2025-03-24 11:46:19.740,2025-03-24 11:46:10.177


**Finding:** No records were identified where `TimeCallEnded` occurred before `TimeCallStart`. One record was identified where `TimeCallConfirmRPC` occurred after `TimeCallEnded` by approximately 9.6 seconds. The affected record was an inbound call with a recorded RPC and otherwise plausible timestamps. Because this is an isolated case and the difference may reflect the way the call system records RPC confirmation separately from call termination, the record will be retained without modifying the timestamps.


The timestamp fields will also be checked for their underlying data types. Datetime fields should use pandas datetime types so that chronological validation and subsequent feature engineering can be performed reliably.


In [174]:
call_history[
    ["CallDate", "TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]
].dtypes

CallDate              datetime64[us]
TimeCallStart                    str
TimeCallConfirmRPC               str
TimeCallEnded                    str
dtype: object

In [175]:
for col in ["TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]:
    print(col)
    print(call_history[col].dropna().head())
    print()

TimeCallStart
0    2025-02-06 08:27:30.053
1    2025-02-06 08:28:11.107
2    2025-02-06 08:29:13.637
3    2025-02-06 08:29:40.383
4    2025-02-06 11:55:22.893
Name: TimeCallStart, dtype: str

TimeCallConfirmRPC
4     2025-02-06 11:57:00.150
6     2025-02-07 12:53:22.823
7     2025-02-10 08:06:46.030
10    2025-02-10 14:47:41.413
19    2025-02-13 14:44:47.370
Name: TimeCallConfirmRPC, dtype: str

TimeCallEnded
0    2025-02-06 08:27:42.980
1    2025-02-06 08:28:36.377
2    2025-02-06 08:29:17.713
3    2025-02-06 08:30:03.303
4    2025-02-06 11:57:18.570
Name: TimeCallEnded, dtype: str



In [176]:
for col in ["TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]:
    call_history[col] = pd.to_datetime(call_history[col], errors="coerce")

In [177]:
call_history[
    ["CallDate", "TimeCallStart", "TimeCallConfirmRPC", "TimeCallEnded"]
].dtypes

CallDate              datetime64[us]
TimeCallStart         datetime64[us]
TimeCallConfirmRPC    datetime64[us]
TimeCallEnded         datetime64[us]
dtype: object

In [178]:
call_history[
    call_history["TimeCallStart"].notna() &
    (call_history["CallDate"].dt.normalize() != call_history["TimeCallStart"].dt.normalize())
][
    ["M_IDX", "HistoryID", "CallDate", "TimeCallStart"]
].shape

(0, 4)

### 7.3.2 Timestamp Data Types and Date Consistency

The call timestamp fields were initially stored as strings despite containing consistently formatted datetime values. `TimeCallStart`, `TimeCallConfirmRPC`, and `TimeCallEnded` were converted to pandas datetime types to support chronological validation and subsequent feature engineering.

The recorded `CallDate` was then compared with the calendar date of `TimeCallStart` for all records where `TimeCallStart` was available.

**Finding:** All records with a recorded `TimeCallStart` have a `CallDate` matching the calendar date of the start timestamp. No inconsistencies were identified, so no date corrections are required.


In [179]:
call_history.loc[
    call_history["CallDate"] < pd.Timestamp("2025-02-01"),
    ["M_IDX", "HistoryID", "CallDate", "CallType", "CallTypeID"]
].shape

(63, 5)